In [ ]:
from supabase import create_client
from dotenv import load_dotenv
import os, pandas as pd
import logging

load_dotenv()
supabase = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))
df_orders = pd.DataFrame(supabase.table('sales').select('*').execute().data)
df_customers = pd.DataFrame(supabase.table('customers').select('*').execute().data)
print(f"Tellimusi: {len(df_orders)}, Kliente: {len(df_customers)}")
print("--- TELLIMUSTE INFO ---")
df_orders.info()

print("\n--- KLIENDIDE INFO ---")
df_customers.info()

In [ ]:
# 1. Veendume, et logger on seadistatud
logger = logging.getLogger(__name__)
logger.info("Alustame Tartu linna tellimuste analüüsi...")

try:
    # 2. Ühendame baasandmed (Left join)
    df_merged = df_orders.merge(df_customers[['customer_id', 'city']], on='customer_id', how='left')
    
    # 3. Filtreerime välja ainult TARTU tellimused (koos tühikute ja suurtähtede kaitsega)
    # Kui tahad hoopis Pärnut, asenda 'tartu' lihtsalt sõnaga 'pärnu'
    df_tartu = df_merged[df_merged['city'].str.strip().str.lower() == 'tartu'].copy()
    
    if df_tartu.empty:
        logger.warning("Andmebaasis ei leitud Tartu linna kohta ühtegi tellimust!")
    else:
        # 4. ARVUTUSED: tellimuste arv, kogukäive ja keskmine tellimus
        tartu_count = len(df_tartu)
        tartu_revenue = df_tartu['total_price'].sum()
        tartu_average = df_tartu['total_price'].mean() # .mean() leiab keskmise
        
        # 5. Tulemuste väljastamine loggeri kaudu
        logger.info(f"Tartu analüüs lõpetatud edukalt.")
        logger.info(f">>> Tellimuste arv Tartus: {tartu_count} tk")
        logger.info(f">>> Kogukäive Tartus: {tartu_revenue:.2f} EUR")
        logger.info(f">>> Keskmine tellimuse summa Tartus: {tartu_average:.2f} EUR")
        
        # Sorteerime ja kuvame ka visuaalselt Tartu TOP 3 tellimused
        print("\nTOP 3 Tartu tellimust:")
        display(df_tartu.sort_values('total_price', ascending=False).head(3))

except KeyError as e:
    logger.error(f"Viga: kontrolli, kas veerud 'city' ja 'total_price' on tabelites olemas! Detailid: {e}")
except Exception as e:
    logger.critical(f"Ootamatu viga Tartu andmete analüüsimisel: {e}")

In [ ]:
logger = logging.getLogger(__name__)
logger.info("Alustan kõigi linnade ülese koondanalüüsi koostamist...")

try:
    # 1. Ühendame müügid ja kliendid
    df_merged = df_orders.merge(df_customers[['customer_id', 'city']], on='customer_id', how='left')
    
    # Puhastame linnade nimed igaks juhuks tühikutest ja teeme esisuurtäheks (nt "tallinn" -> "Tallinn")
    df_merged['city'] = df_merged['city'].str.strip().str.title()
    
    # 2. Grupeerime LINNA järgi ja arvutame igale linnale korraga kolm näitajat
    city_summary = df_merged.groupby('city').agg(
        tellimuste_arv=('total_price', 'count'),      # Mitu rida/tellimust
        kogukaive=('total_price', 'sum'),             # Summa kokku
        keskmine_tellimus=('total_price', 'mean')     # Keskmine tehing (.mean)
    ).reset_index()
    
    # Sorteerime kogukäibe järgi, et kõige suurem linn oleks eespool
    city_summary = city_summary.sort_values('kogukaive', ascending=False)
    
    logger.info("Kõigi linnade koondtabel on edukalt arvutatud!")
    
    # Kuvame ilusa koondtabeli
    print("\n LINNADE VÕRDLUSTABEL:")
    display(city_summary)

except Exception as e:
    logger.error(f"Viga linnade koondtabeli arvutamisel: {e}")

In [ ]:
logger = logging.getLogger(__name__)
logger.info("Alustan suurte tellimuste (kliendivaalade) analüüsi...")

try:
    # ÄRIKÜSIMUS: Kes teevad meil tellimusi väärtusega üle 100€ ja mis on nende keskmine ost?
    # 1. Filtreerime välja ainult tellimused, mille hind on suurem kui 100€
    df_large_orders = df_orders[df_orders['total_price'] > 100].copy()
    
    # 2. Kontrollime tulemusi (Vastavalt kontrolltabelile: shape, head, describe)
    logger.info("Andmete filtreerimine edukas. Alustan tulemuste kontrolli.")
    
    print("\n=== 1. ANDMETABELI KUJU (shape) ===")
    print(f"Suuri tellimusi leiti: {df_large_orders.shape[0]} tükki (veerge: {df_large_orders.shape[1]})")
    
    print("\n=== 2. ESIMESED READ (head) ===")
    display(df_large_orders.head(3))
    
    print("\n=== 3. STATISTILINE KOKKUVÕTE (describe) ===")
    display(df_large_orders[['total_price']].describe())

except Exception as e:
    logger.error(f"Päringu käivitamisel tekkis viga: {e}")

In [ ]:
from datetime import datetime

def weekly_sales_report(df, report_date=None):
    """Genereeri iganädalane müügiraport.

    Args:
        df: DataFrame müügitellimustega
        report_date: Raporti kuupäev (vaikimisi täna)
    Returns:
        dict: Raporti kokkuvõte
    """
    if report_date is None:
        report_date = datetime.now().strftime('%Y-%m-%d')
    return {
        'report_date': report_date,
        'total_orders': len(df),
        'total_revenue': round(df['total_price'].sum(), 2),
        'avg_order': round(df['total_price'].mean(), 2),
    }

# Käivita
result = weekly_sales_report(df_orders)
for key, value in result.items():
    print(f"  {key}: {value}")

In [ ]:
def calculate_rfm(df, reference_date=None):
    """Arvuta RFM skoorid ja segmendid.

    Args:
        df: DataFrame tellimustega (veerud: customer_id, sale_date, total_price)
        reference_date: Viitekuupäev Recency arvutamiseks

    Returns:
        DataFrame: RFM skoorid ja segmendid iga kliendi kohta
    """
    if reference_date is None:
        reference_date = pd.to_datetime('today')
    else:
        reference_date = pd.to_datetime(reference_date)

    df['sale_date'] = pd.to_datetime(df['sale_date'])

    # Recency: päevi viimasest ostust
    recency = df.groupby('customer_id')['sale_date'].max().reset_index()
    recency.columns = ['customer_id', 'last_purchase']
    recency['recency_days'] = (reference_date - recency['last_purchase']).dt.days

    # Frequency: ostude arv
    frequency = df.groupby('customer_id').size().reset_index(name='frequency')

    # Monetary: kogukulutus
    monetary = df.groupby('customer_id')['total_price'].sum().reset_index()  # Täida!
    monetary.columns = ['customer_id', 'monetary']

    # Liida kokku
    rfm = recency[['customer_id', 'recency_days']].merge(
        frequency, on='customer_id'
    ).merge(
        monetary, on='customer_id'
    )

    # Skooride määramine (lihtsustatud)
    rfm['R_score'] = pd.qcut(rfm['recency_days'], q=3, labels=[3, 2, 1]).astype(int)
    rfm['F_score'] = pd.qcut(
        rfm['frequency'].rank(method='first'), q=3, labels=[1, 2, 3]
    ).astype(int)
    rfm['M_score'] = pd.qcut(rfm['monetary'], q=3, labels=[1, 2, 3]).astype(int)
    rfm['RFM_score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

    # Segmenteerimine
    def assign_segment(score):
        if score >= 8:
            return 'VIP Champions'              # Täida! Vihje: 'VIP Champions'
        elif score >= 6:
            return 'Active Customers'              # Täida!
        elif score >= 4:
            return 'At Risk Customers'              # Täida!
        else:
            return 'Lost Customers'              # Täida!

    rfm['segment'] = rfm['RFM_score'].apply(assign_segment)
    return rfm

# Testi
rfm_result = calculate_rfm(df_orders, reference_date='2024-08-01')
print(rfm_result.sort_values('RFM_score', ascending=False))
print(f"\nSegmentide jaotus:")
print(rfm_result['segment'].value_counts())